# Kadai 3
Lyapunov exponents — Logistic map, Hénon map, Lorenz system.

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Lyapunov Exponent — Logistic Map
$$x_{n+1} = f(x_n) = ax_n(1-x_n)$$
$$\lambda = \lim_{N\to\infty} \frac{1}{N}\sum_{n=0}^{N-1} \ln|f'(x_n)|, \quad f'(x) = a(1-2x)$$

In [2]:
def lyapunov_logistic(a, N=100000, transient=1000, x0=0.5):
    x = x0
    for _ in range(transient):
        x = a * x * (1 - x)
    total = 0.0
    for _ in range(N):
        deriv = abs(a * (1 - 2*x))
        if deriv == 0:
            break
        total += np.log(deriv)
        x = a * x * (1 - x)
    return total / N

a_vals = [3.58, 3.67, 3.86, 4.0]
print('Logistic map Lyapunov exponents:')
print('-' * 30)
for a in a_vals:
    lam = lyapunov_logistic(a)
    print(f'  a = {a:.2f}:  λ = {lam:.6f}')

Logistic map Lyapunov exponents:
------------------------------
  a = 3.58:  λ = 0.104710
  a = 3.67:  λ = 0.307448
  a = 3.86:  λ = 0.383747
  a = 4.00:  λ = 1.386294


## 2. Lyapunov Spectrum — Hénon Map
$$x_{n+1} = 1 - ax_n^2 + y_n, \quad y_{n+1} = bx_n$$
Jacobian: $J = \begin{pmatrix} -2ax_n & 1 \\ b & 0 \end{pmatrix}$

Use QR decomposition (Gram-Schmidt) to extract both exponents.

In [3]:
def lyapunov_henon(a=1.4, b=0.3, N=100000, transient=1000):
    x, y = 0.1, 0.1
    for _ in range(transient):
        x, y = 1 - a*x**2 + y, b*x

    # tangent vectors stored as columns of Q (2x2 identity to start)
    Q = np.eye(2)
    log_sum = np.zeros(2)

    for _ in range(N):
        J = np.array([[-2*a*x, 1],
                      [b,      0]])
        x, y = 1 - a*x**2 + y, b*x

        M = J @ Q
        Q, R = np.linalg.qr(M)
        log_sum += np.log(np.abs(np.diag(R)))

    return log_sum / N

lams = lyapunov_henon()
print('Hénon map Lyapunov spectrum (a=1.4, b=0.3):')
print('-' * 40)
for i, lam in enumerate(sorted(lams, reverse=True)):
    print(f'  λ_{i+1} = {lam:.6f}')
print(f'  Sum    = {sum(lams):.6f}  (should ≈ ln|b| = {np.log(0.3):.6f})')

Hénon map Lyapunov spectrum (a=1.4, b=0.3):
----------------------------------------
  λ_1 = 0.418469
  λ_2 = -1.622442
  Sum    = -1.203973  (should ≈ ln|b| = -1.203973)


## 3. Lyapunov Spectrum — Lorenz System
$$\dot{x}=\sigma(y-x),\quad \dot{y}=x(r-z)-y,\quad \dot{z}=xy-bz$$
$\sigma=10,\; r=28,\; b=8/3$

Integrate state + tangent space (9 tangent vectors) simultaneously, renormalize via QR at each step.

In [4]:
def lorenz_jacobian(x, y, z, sigma=10, r=28, b=8/3):
    return np.array([
        [-sigma,  sigma, 0],
        [r - z,  -1,   -x],
        [y,       x,   -b]
    ])

def lorenz_tangent_rhs(state, Q, sigma=10, r=28, b=8/3):
    """Returns dstate/dt and dQ/dt."""
    x, y, z = state
    dstate = np.array([sigma*(y-x), x*(r-z)-y, x*y-b*z])
    J = lorenz_jacobian(x, y, z, sigma, r, b)
    dQ = J @ Q   # tangent evolution
    return dstate, dQ

def lyapunov_lorenz(sigma=10, r=28, b=8/3, T=200, dt=0.01, t_trans=20):
    # transient
    state = np.array([1.0, 1.0, 1.0])
    n_trans = int(t_trans / dt)
    for _ in range(n_trans):
        k1 = np.array([sigma*(state[1]-state[0]),
                       state[0]*(r-state[2])-state[1],
                       state[0]*state[1]-b*state[2]])
        state = state + dt * k1  # simple Euler for transient (enough)

    # use RK4 for main integration
    def f(s):
        x, y, z = s
        return np.array([sigma*(y-x), x*(r-z)-y, x*y-b*z])

    Q = np.eye(3)
    log_sum = np.zeros(3)
    n_steps = int(T / dt)

    for _ in range(n_steps):
        x, y, z = state
        J = lorenz_jacobian(x, y, z, sigma, r, b)

        # RK4 for state
        k1 = f(state)
        k2 = f(state + 0.5*dt*k1)
        k3 = f(state + 0.5*dt*k2)
        k4 = f(state + dt*k3)
        state = state + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

        # Euler for tangent (cheaper; fine for spectrum)
        J_new = lorenz_jacobian(*state, sigma, r, b)
        Q = Q + dt * (J_new @ Q)

        Q, R = np.linalg.qr(Q)
        log_sum += np.log(np.abs(np.diag(R)))

    return log_sum / T

lams = lyapunov_lorenz()
lams_sorted = sorted(lams, reverse=True)
print('Lorenz system Lyapunov spectrum (σ=10, r=28, b=8/3):')
print('-' * 50)
for i, lam in enumerate(lams_sorted):
    print(f'  λ_{i+1} = {lam:.6f}')
print(f'  Sum    = {sum(lams):.6f}  (should ≈ -(σ+1+b) = {-(10+1+8/3):.6f})')
print(f'  Kaplan-Yorke dim ≈ {2 + lams_sorted[0]/(abs(lams_sorted[2])):.4f}')

Lorenz system Lyapunov spectrum (σ=10, r=28, b=8/3):
--------------------------------------------------
  λ_1 = 1.187923
  λ_2 = 0.516646
  λ_3 = -15.879298
  Sum    = -14.174729  (should ≈ -(σ+1+b) = -13.666667)
  Kaplan-Yorke dim ≈ 2.0748
